<div style="font-family: 'Times New Roman', serif;">

[Ordinary Least Sqares estimator]

$$Y=\beta X + \varepsilon$$
where $Y \in \mathbb{R}^{n\times 1}$, $X \in \mathbb{R}^{n\times p}$, $Y \in \mathbb{R}^{p\times 1}$
and $\varepsilon \sim \mathcal{N}(0,\sigma^2 I_n)$ : error vector with mean zero and covariance matrix $I_n$

$$ X = \begin{bmatrix} x_{11} & x_{12} & ... & x_{1p}
                    \\ x_{21} & x_{22} & ... & x_{2p}
                    \\ ...    & ...    & ... & ...
                    \\ x_{n1} & x_{n2} & ... & x_{np}
                    \end{bmatrix} \in \mathbb{R}^{n\times p}$$



**1) Estimating the parameters**  

Our goal is to find $\beta$ that minimized $S(\beta)$ 
$$S(\beta) = \|Y-X\beta\|^2 = (Y-X\beta)^\top (Y-X\beta) = Y^\top Y - \beta^\top X^\top Y - Y^\top X\beta +\beta^\top X^\top X\beta$$
$$\nabla_\beta S(\beta) = - X^\top Y - (Y^\top X)^\top + 2(X^\top X)\beta = -2 X^\top Y + 2(X^\top X)\beta$$
Therefore, $\nabla_\beta S(\beta) = 0$ when
$$\hat{\beta} = (X^\top X)^{-1}X^\top Y$$

</div>

In [53]:
import numpy as np

np.random.seed(42)

n = 20
p = 5

# true parameter
true_beta = np.array([3.0, 1.5, -2.0, 5.2, -3.2, 2.1])

# independent variables matrix
X_features = np.random.randn(n, p)
# add intercept column
X = np.column_stack([np.ones(n), X_features])

# error vector
eps = np.random.randn(n)

# * : element-wise multiplication,  @ : matrix multiplication
# Creating a dependent variable vector
Y = X @ true_beta + eps

# OLS : beta_hat = (X.T @ X)^(-1) @ X.T @ Y
XTX = X.T @ X
XTX_inv = np.linalg.inv(XTX) # inverting matrix
beta_hat = XTX_inv @ X.T @ Y

print("=== OLS Result ===")
print(f"true beta: {true_beta}")
print(f"beta hat: {np.round(beta_hat,3)}")

=== OLS Result ===
true beta: [ 3.   1.5 -2.   5.2 -3.2  2.1]
beta hat: [ 2.844  1.76  -1.614  5.275 -3.53   2.015]


<div style="font-family: 'Times New Roman', serif;">

**2) Estimating the variance of residuals**  

Let $\mathbb{e} = Y - X\hat{\beta}$ be a residual vector (estimation error vector)

$$ \text{Residual Sum of Squares, RSS : }\mathbb{e}^\top \mathbb{e} = \sum_{i=1}^n e_i^2$$

as the degree of freedom is $n - (p+1)$ (since residuals have p+1 constraints : sum of residuals must be 0 and residuals must be independent to each features)  
instead of dividing RSS by $n$, we divide by $n-(p+1)$ to estimate the variance of $e_i$
$$s^2 = \hat{\sigma}^2 = \frac{\sum_{i=1}^ke_i^2}{n-k}=\frac{e^\top e}{n - (p+1)}$$

</div>

In [54]:

residuals = Y - X @ beta_hat
sigma2_hat = (residuals.T @ residuals) / (n - (p+1))

print(f"Y hat    : {np.round(X @ beta_hat, 4)}")
print(f"Y        : {np.round(Y, 4)}")
print(f"residuals: {np.round(residuals,3)}\n")
print(f"=== Variance of the residual ===")
print(f"sigma2 hat: {np.round(sigma2_hat, 4)}\n")


Y hat    : [ 1.5098  6.6802  7.3343  5.5065 10.0751  8.4109  4.1136 -4.8941  1.3404
  3.132   0.3829  5.002   2.3234  8.0635 -9.9926 -1.756   9.8765  5.1207
 -1.886   0.6767]
Y        : [  0.6088   5.7021   6.652    4.9539   9.2559   9.1355   5.3209  -4.6011
   1.2798   3.4621  -0.9747   5.4295   2.494   10.348  -10.421   -1.6621
   9.5672   3.7305  -1.0744   1.8129]
residuals: [-0.901 -0.978 -0.682 -0.553 -0.819  0.725  1.207  0.293 -0.061  0.33
 -1.358  0.428  0.171  2.285 -0.428  0.094 -0.309 -1.39   0.812  1.136]

=== Variance of the residual ===
sigma2 hat: 1.2026



<div style="font-family: 'Times New Roman', serif;">

**3) Estimating the Variance-Covariance matrix of the estimated parameter (using estimated variance of residuals)**  

$$\hat{\beta} = (X^\top X)^{-1}X^\top Y = (X^\top X)^{-1}X^\top (X\beta + \varepsilon) = \underbrace{(X^\top X)^{-1}X^\top X}_{I}\beta + (X^\top X)^{-1}X^\top \varepsilon = \beta + (X^\top X)^{-1}X^\top \varepsilon$$

*General Rule for the Variance of an Affine Transformation : $Var(AZ+c) = AVar(Z)A^\top$ for any random vector $Z$ and non-random matrix $A$ and constant vector $c$*  
</br>
Therefore, 
$$Var(\hat{\beta}) = Var(\beta + \underbrace{(X^\top X)^{-1} X^\top}_{A} \underbrace{\varepsilon}_{Z}) 
                   = [(X^\top X)^{-1}X^\top] \underbrace{\sigma^2 I_n}_{Var(\varepsilon)} [(X^\top X)^{-1}X^\top]^\top
                   = \sigma^2(X^\top X)^{-1} X^\top X [\underbrace{(X^\top X)^{-1}}_{\text{symmetric}}]^\top 
                   = \sigma^2(X^\top X)^{-1} \underbrace {X^\top X (X^\top X)^{-1}}_{I}
                   = \sigma^2(X^\top X)^{-1}$$  

</br>  

Since $\sigma$ is unknown, plugging $\hat{\sigma}^2$ yields  
$$\widehat{Var}(\hat{\beta}) = \hat{\sigma}^2(X^\top X)^{-1} = \begin{bmatrix} \widehat{Var}(\hat{\beta_0})              & \widehat{Cov}(\hat{\beta_0}\hat{\beta_1}) & ... & \widehat{Cov}(\hat{\beta_0}\hat{\beta_p}) \\
                                                                               \widehat{Cov}(\hat{\beta_1}\hat{\beta_0}) & \widehat{Var}(\hat{\beta_1})              & ... & \widehat{Cov}(\hat{\beta_1}\hat{\beta_p}) \\
                                                                               ...                                       & ...                                       & ... & ... \\
                                                                               \widehat{Cov}(\hat{\beta_p}\hat{\beta_0}) & \widehat{Cov}(\hat{\beta_p}\hat{\beta_1}) & ... & \widehat{Var}(\hat{\beta_p})
                                                                               \end{bmatrix}
$$  

$$diag(\widehat{Var}(\hat{\beta})) = [\widehat{Var}(\hat{\beta_0}), \widehat{Var}(\hat{\beta_1}), ..., \widehat{Var}(\hat{\beta_p})]^\top$$

**4) Calculating the standard error of an estimator**   

$$SE(\hat{\beta}_j) = \sqrt{\widehat{Var}(\hat{\beta}_j)}$$

</div>

In [55]:

cov_beta_hat = sigma2_hat * XTX_inv

se_beta_hat = np.sqrt(np.diag(cov_beta_hat))

print(f"=== Variance of estimated parameter (beta hat) ===")
print(f"true beta     : {true_beta}")
print(f"beta hat      : {np.round(beta_hat,3)}")
print(f"Standard Error: {np.round(se_beta_hat, 3)}")

=== Variance of estimated parameter (beta hat) ===
true beta     : [ 3.   1.5 -2.   5.2 -3.2  2.1]
beta hat      : [ 2.844  1.76  -1.614  5.275 -3.53   2.015]
Standard Error: [0.294 0.341 0.306 0.34  0.301 0.245]


<div style="font-family: 'Times New Roman', serif;">
  

1. Null Hypothesis ($H_0$)

$$H_0 : \beta_j = 0 \quad$$
null hypothesis: the feature has no linear effect on $Y$. 

2. t-statistics (Signal-to-Noise Ratio) : how many standard deviations $\hat{\beta}_j$ is away from 0

$$t = \frac{\hat{\beta}_j - 0}{SE(\hat{\beta}_j)} = \frac{\text{Observed Effect}}{\text{Estimation Uncertainty}}$$

3. p-value : assuming null hypothesis, probability of observing a t-stat equal or more extreme than the observed data's
$$p\text{-value} = \mathbb{P}(|T|\geq |t_{observed}| \mid H_0 \text{ is true}) = 2\int_{|t_{observed}|}^{\infty} f(t;v) dt = 2(1-\mathbb{P}(X\leq x))$$
*f is the probability density function of the t-distribution*  


4. confidence interval 

Critical Value :

$$CV = g^{-1}\left(1-\frac{\alpha}{2} ; v\right)  \quad \text{where } g(t;v) = \int_{-\infty}^{t} f(u;v) du$$

</br>  

Since $\frac{\hat{\beta}_j - \beta_j}{SE(\hat{\beta}_j)}$ $\sim$ t-distribution (assuming $\varepsilon \sim \mathcal{N}(0,\sigma^2 I_n)$)
$$\mathbb{P}\left(-CV\leq \frac{\hat{\beta}_j - \beta_j}{SE(\hat{\beta}_j)} \leq CV \right) = 1-\alpha$$
$$\mathbb{P}\left( \hat{\beta}_j -CV\cdot SE(\hat{\beta}_j) \leq \beta_j \leq \hat{\beta}_j +CV\cdot SE(\hat{\beta}_j) \right) = 1-\alpha$$

Interval $[\hat{\beta}_j -CV\cdot SE(\hat{\beta}_j), \hat{\beta}_j +CV\cdot SE(\hat{\beta}_j)]$ is the set of all values s.t. null hypothesis can't be rejected at significance level $\alpha$

</div>

In [64]:
from scipy import stats

# null hypothesis: H0: beta_hat = 0  (feature has no effect on Y)
# t-statistics : t = (beta_hat - 0) / SE(beta_hat)
t_stats = beta_hat / se_beta_hat

# t : t-statistics for the hypothesis test
# sf : survival function (1 - CDF) 
p_values = 2 * stats.t.sf(np.abs(t_stats), df = n-(p+1))



# alpha : significance level for the hypothesis test
# ppf : percent point function (inverse of CDF) 
# t_crit : critical value of the t-distribution for the given significance level and degrees of freedom
alpha = 0.05
t_crit = stats.t.ppf(1 - alpha / 2, df = n-(p+1))

ci_lower = beta_hat - t_crit * se_beta_hat
ci_upper = beta_hat + t_crit * se_beta_hat


for i in range(len(beta_hat)):
    print(f"[beta_{i} : {round(beta_hat[i],3)}]\n")

    print(f"=== Hypothesis Testing (alpha = {alpha}) ===")
    print(f"Hypothesis Test: t = {t_stats[i]:.3f}, p = {p_values[i]:.3f}, reject H0: {p_values[i] < alpha}")
    print("\n")
    print(f"=== Confidence Intervals (alpha = {alpha}) ===")
    print(f"Confidence Interval: [{ci_lower[i]:.3f}, {ci_upper[i]:.3f}]")
    print("\n")


[beta_0 : 2.844]

=== Hypothesis Testing (alpha = 0.05) ===
Hypothesis Test: t = 9.658, p = 0.000, reject H0: True


=== Confidence Intervals (alpha = 0.05) ===
Confidence Interval: [2.212, 3.475]


[beta_1 : 1.76]

=== Hypothesis Testing (alpha = 0.05) ===
Hypothesis Test: t = 5.161, p = 0.000, reject H0: True


=== Confidence Intervals (alpha = 0.05) ===
Confidence Interval: [1.028, 2.491]


[beta_2 : -1.614]

=== Hypothesis Testing (alpha = 0.05) ===
Hypothesis Test: t = -5.270, p = 0.000, reject H0: True


=== Confidence Intervals (alpha = 0.05) ===
Confidence Interval: [-2.271, -0.957]


[beta_3 : 5.275]

=== Hypothesis Testing (alpha = 0.05) ===
Hypothesis Test: t = 15.508, p = 0.000, reject H0: True


=== Confidence Intervals (alpha = 0.05) ===
Confidence Interval: [4.545, 6.004]


[beta_4 : -3.53]

=== Hypothesis Testing (alpha = 0.05) ===
Hypothesis Test: t = -11.715, p = 0.000, reject H0: True


=== Confidence Intervals (alpha = 0.05) ===
Confidence Interval: [-4.176, -2.883]